# N095 · 收支平衡与双向约束

**目标：** 用前缀亏损和两侧约束推导贪心。

**先修：** N031, N093。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 前缀最小；候选起点排除；两遍扫描；局部约束合并。

## 从问题到算法

加油站问题考察连续收支：从候选起点到i累计为负，则这段中任何后续位置都不能作为起点挽救这段失败。糖果问题则有来自左右的两组不等式；分别求满足一侧约束的最小量，再逐点取最大，不能只单向扫描。

## 最小实现

**本章接口：** `can_complete_circuit(gas, cost)`、`candy(ratings)`

In [1]:
def can_complete_circuit(gas,cost):
    if len(gas)!=len(cost) or not gas: return -1
    total=tank=start=0
    for i,(g,c) in enumerate(zip(gas,cost)):
        total+=g-c; tank+=g-c
        if tank<0: start=i+1; tank=0
    return start if total>=0 else -1

def candy(ratings):
    n=len(ratings); left=[1]*n; right=1; total=0
    for i in range(1,n):
        if ratings[i]>ratings[i-1]: left[i]=left[i-1]+1
    for i in range(n-1,-1,-1):
        right=right+1 if i<n-1 and ratings[i]>ratings[i+1] else 1
        total+=max(left[i],right)
    return total

## 正确性、前提与复杂度

加油站扫描不变量：当前候选以前的起点已被一个负和区间排除；总和非负时，最后剩余候选可完成一圈。糖果左右数组分别是必要下界，其逐点最大同时满足两侧不等式，故不仅可行，而且逐点最小。

**代价：** 均O(n)时间；加油站O(1)辅助，糖果O(n)辅助。相邻同分不要求相同糖果；空评分总量0。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

ratings=[1,3,4,5,2]; left=[1]*len(ratings); right=[1]*len(ratings)
for i in range(1,len(ratings)):
    if ratings[i]>ratings[i-1]: left[i]=left[i-1]+1
for i in range(len(ratings)-2,-1,-1):
    if ratings[i]>ratings[i+1]: right[i]=right[i+1]+1
show_table(['评分','左约束','右约束','分配'],[(r,a,b,max(a,b)) for r,a,b in zip(ratings,left,right)])

评分,左约束,右约束,分配
1,1,1,1
3,2,1,2
4,3,1,3
5,4,2,4
2,1,1,1


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert can_complete_circuit([1,2,3,4,5],[3,4,5,1,2])==3
assert can_complete_circuit([1],[2])==-1
assert candy([1,0,2])==5
from itertools import product
for ratings in product(range(3),repeat=4):
    feasible=[]
    for values in product(range(1,5),repeat=4):
        if all(ratings[i]==ratings[i+1] or (values[i]>values[i+1] if ratings[i]>ratings[i+1] else values[i]<values[i+1]) for i in range(3)): feasible.append(sum(values))
    assert candy(ratings)==min(feasible)
for differences in product(range(-2,3),repeat=4):
    gas=[max(x,0) for x in differences]; cost=[max(-x,0) for x in differences]
    starts=[s for s in range(4) if all(sum(differences[(s+j)%4] for j in range(k+1))>=0 for k in range(4))]
    got=can_complete_circuit(gas,cost)
    assert (got in starts) if starts else got==-1
print('N095: 所有本章断言通过')

N095: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 证明一次失败能排除一段起点。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 为什么糖果只从左向右扫描不够。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 134. Gas Station（canonical）
- 135. Candy（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。